# LeetCode 3110 — Score of a String bằng Python

Link đề: https://leetcode.com/problems/score-of-a-string/ · Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def scoreOfString(self, s: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **điểm của chuỗi** là **tổng của `|mã ASCII của ký tự này − mã ASCII của ký tự kế tiếp|`** trên mọi cặp ký tự **liền kề**.
2. Nắm các công cụ Python: `ord(c)` (mã của ký tự), `abs` (giá trị tuyệt đối), duyệt **cặp liền kề** bằng **chỉ số** (`range(len(s) - 1)`), bằng **`zip(s, s[1:])`**, bằng **biến `prev`**, và bằng `bytes`.
3. Đi từ cách **duyệt chỉ số và tính từng cặp** → **`zip` với generator** → **một biến `prev` (không cắt chuỗi)** → **chuyển cả chuỗi thành mảng mã rồi dùng `map`/`pairwise`**.
4. Hiểu mấu chốt: `n` ký tự có đúng **`n − 1` cặp liền kề**; không có "vòng tròn" (ký tự cuối không ghép với ký tự đầu), và dấu `abs` đảm bảo mỗi cặp đóng góp **số không âm**.
5. Tránh các lỗi kinh điển: quên `ord` (trừ hai chuỗi gây `TypeError`), quên `abs`, vượt biên chỉ số (`s[i + 1]` khi `i` là cuối), đếm cặp cuối-đầu, và nhầm khoảng cách trong bảng chữ cái với mã ASCII.

## 2. Đề bài & ví dụ

Cho một chuỗi `s`. **Điểm** (score) của `s` được định nghĩa là **tổng các hiệu tuyệt đối giữa mã ASCII của hai ký tự liền kề**.

Hãy trả về điểm của `s`.

**Ví dụ 1**

- Input: `s = "hello"`
- Output: `13`
- Giải thích: mã ASCII của `h, e, l, l, o` lần lượt là `104, 101, 108, 108, 111`. Điểm là `|104 − 101| + |101 − 108| + |108 − 108| + |108 − 111| = 3 + 7 + 0 + 3 = 13`.

**Ví dụ 2**

- Input: `s = "zaz"`
- Output: `50`
- Giải thích: mã ASCII của `z, a, z` là `122, 97, 122`. Điểm là `|122 − 97| + |97 − 122| = 25 + 25 = 50`.

**Ràng buộc:** `2 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
import random
import string
from itertools import pairwise
from operator import sub

examples = [
    ("hello", 13),
    ("zaz", 50),
    ("ab", 1),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", expected)

'hello' -> kỳ vọng 13
'zaz' -> kỳ vọng 50
'ab' -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Mã ASCII và hàm `ord`, `chr`

Máy tính lưu mỗi ký tự dưới dạng một **số** gọi là **mã** (ASCII/Unicode). Với chữ cái thường tiếng Anh, `'a'` có mã `97`, `'b'` là `98`, ..., `'z'` là `122` (liên tiếp, tăng dần). Hai hàm đổi qua lại:

- `ord(c)`: ký tự → mã (số);
- `chr(n)`: mã → ký tự.

Ký tự **không** trừ trực tiếp được (`'e' - 'h'` báo `TypeError`); ta phải đổi sang mã bằng `ord`.

In [2]:
print(ord("a"), ord("b"), ord("z"))        # 97 98 122
print(chr(104), chr(101))                  # h e
print(ord("h") - ord("e"))                 # 3
try:
    "h" - "e"
except TypeError as e:
    print("TypeError:", e)

97 98 122
h e
3
TypeError: unsupported operand type(s) for -: 'str' and 'str'


### 3.2. Giá trị tuyệt đối `abs`

`abs(x)` trả về `x` nếu `x >= 0` và `-x` nếu `x < 0`; luôn **không âm**. Ở đây `|104 − 101| = 3` và `|101 − 108| = 7` (không phải `−7`).

In [3]:
print(abs(3), abs(-7), abs(0))             # 3 7 0
print(abs(ord("e") - ord("l")))            # 7

3 7 0
7


### 3.3. Cặp liền kề: chỉ số `i` và `i + 1`

Chuỗi `s` dài `n` có chỉ số `0..n-1`. Cặp liền kề là `(s[i], s[i + 1])` với `i` từ `0` tới **`n − 2`**. Vì vậy `range(len(s) - 1)`. Nếu cho `i` chạy tới `n − 1` thì `s[i + 1]` **vượt biên** (`IndexError`).

In [4]:
s = "hello"
for i in range(len(s) - 1):
    print(i, s[i], s[i + 1])
print("số cặp =", len(s) - 1)
try:
    s[len(s)]
except IndexError as e:
    print("IndexError:", e)

0 h e
1 e l
2 l l
3 l o
số cặp = 4
IndexError: string index out of range


### 3.4. `zip(s, s[1:])`: ghép mỗi ký tự với ký tự kế tiếp

`s[1:]` là `s` bỏ ký tự đầu. `zip(s, s[1:])` ghép phần tử cùng vị trí của hai dãy, tạo ra đúng các **cặp liền kề** và **tự dừng** khi một dãy hết, nên không lo vượt biên.

In [5]:
s = "hello"
print(s[1:])                       # ello
print(list(zip(s, s[1:])))         # [('h','e'), ('e','l'), ('l','l'), ('l','o')]

ello
[('h', 'e'), ('e', 'l'), ('l', 'l'), ('l', 'o')]


### 3.5. Tổng bằng `sum` và generator expression

`sum(biểu_thức for ... in ...)` cộng các giá trị biểu thức sinh ra, không tạo list trung gian. Dạng này rất hay dùng để "tổng của mọi cặp".

In [6]:
print(sum(x * x for x in [1, 2, 3]))                     # 14
s = "hello"
print(sum(abs(ord(a) - ord(b)) for a, b in zip(s, s[1:])))   # 13

14
13


### 3.6. `bytes`, `itertools.pairwise`, `operator.sub`

- `s.encode()` biến chuỗi thành **bytes**; duyệt `bytes` cho ra **các số nguyên** (mã) — với chữ thường tiếng Anh, mã trùng ASCII, nên không cần `ord`.
- `itertools.pairwise(x)` (Python 3.10+) cho các cặp liền kề `(x0, x1), (x1, x2), ...` — tương đương `zip(x, x[1:])` nhưng chạy trên mọi iterable.
- `operator.sub(a, b)` là hàm tương đương `a - b`, hữu ích với `map`.

In [7]:
data = "hello".encode()
print(list(data))                          # [104, 101, 108, 108, 111]
print(list(pairwise(data)))                # [(104, 101), (101, 108), (108, 108), (108, 111)]
print(sub(10, 3), list(map(sub, [5, 9], [1, 4])))   # 7 [4, 5]

[104, 101, 108, 108, 111]
[(104, 101), (101, 108), (108, 108), (108, 111)]
7 [4, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "hello"`. Viết mã của từng ký tự và các cặp liền kề:

| chỉ số | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| ký tự | h | e | l | l | o |
| mã | 104 | 101 | 108 | 108 | 111 |

Các cặp liền kề và đóng góp:

| cặp | hiệu | `abs` |
|---|---|---|
| (h, e) | 104 − 101 = 3 | 3 |
| (e, l) | 101 − 108 = −7 | 7 |
| (l, l) | 108 − 108 = 0 | 0 |
| (l, o) | 108 − 111 = −3 | 3 |

Tổng `3 + 7 + 0 + 3 = 13`. Bài toán chỉ gồm: **đổi sang mã → lấy hiệu từng cặp liền kề → `abs` → cộng**. Ký tự `o` ở cuối **không** ghép với `h` ở đầu (không có vòng tròn).

In [8]:
s = "hello"
codes = [ord(c) for c in s]
print("mã:", codes)
diffs = [abs(codes[i] - codes[i + 1]) for i in range(len(codes) - 1)]
print("các |hiệu|:", diffs, "-> tổng =", sum(diffs))

mã: [104, 101, 108, 108, 111]
các |hiệu|: [3, 7, 0, 3] -> tổng = 13


## 5. Cách 1 — Brute Force / baseline: vòng lặp chỉ số, tính từng cặp

**Ý tưởng.** Cho `i` chạy `0 .. n − 2`; mỗi lần cộng `abs(ord(s[i]) − ord(s[i + 1]))` vào biến `total`. Đây là cách "dịch đề ra code" trực tiếp nhất.

In [9]:
def score_index(s: str) -> int:
    total = 0
    for i in range(len(s) - 1):                    # n - 1 cặp liền kề
        total += abs(ord(s[i]) - ord(s[i + 1]))    # đổi sang mã, lấy hiệu tuyệt đối
    return total


for s, expected in examples:
    print(repr(s), score_index(s), "(kỳ vọng", expected, ")")

'hello' 13 (kỳ vọng 13 )
'zaz' 50 (kỳ vọng 50 )
'ab' 1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Vòng lặp `n − 1` lần, mỗi lần `O(1)` ⇒ **Time `O(n)`**, **Space `O(1)`**.

**Nhận xét:** về độ phức tạp cách này đã tối ưu (phải đọc từng ký tự ít nhất một lần). Các cách sau chỉ khác về **cách viết** (ngắn gọn hơn, ít sai chỉ số hơn).

## 6. Ý tưởng thuật toán / bước đột phá

Ba "mẹo viết" để bớt lỗi chỉ số và gọn hơn:

1. **`zip(s, s[1:])`**: ghép mỗi ký tự với ký tự kế tiếp, **không cần `range`, không lo vượt biên**.
2. **Nhớ ký tự trước (`prev`)**: duyệt một lần, mỗi bước so ký tự hiện tại với `prev` rồi cập nhật `prev`; không cắt chuỗi (không tốn `O(n)` bộ nhớ cho `s[1:]`).
3. **Làm việc trên mã sẵn**: đổi cả chuỗi thành dãy số (`bytes`/list `ord`) một lần, rồi chỉ còn bài toán "tổng hiệu tuyệt đối của các cặp liền kề của một dãy số" — có thể dùng `map`, `pairwise`.

Cả ba đều là `O(n)`; chọn cách đọc dễ nhất với bạn.

## 7. Các cách tiếp theo

### Cách 2 — `zip` và generator expression

**Giải thích.** `zip(s, s[1:])` cho từng cặp `(a, b)` liền kề; biểu thức `abs(ord(a) - ord(b))` tính đóng góp; `sum(...)` cộng lại. Một dòng.

In [10]:
def score_zip(s: str) -> int:
    return sum(abs(ord(a) - ord(b)) for a, b in zip(s, s[1:]))


for s, expected in examples:
    print(repr(s), score_zip(s), "(kỳ vọng", expected, ")")

'hello' 13 (kỳ vọng 13 )
'zaz' 50 (kỳ vọng 50 )
'ab' 1 (kỳ vọng 1 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionZip:
    def scoreOfString(self, s: str) -> int:
        return sum(abs(ord(a) - ord(b)) for a, b in zip(s, s[1:]))


print(SolutionZip().scoreOfString("zaz"))      # 50

50


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** cho `s[1:]` (một chuỗi mới); nếu muốn `O(1)` dùng Cách 3.

### Cách 3 — Một biến `prev`, không cắt chuỗi

**Giải thích.** Gán `prev = ord(s[0])` rồi duyệt `s[1:]`... để tránh tạo chuỗi mới ta duyệt bằng chỉ số `i` từ `1`: `cur = ord(s[i])`, cộng `abs(cur − prev)`, rồi `prev = cur`. Mỗi ký tự được đổi sang mã **đúng một lần** (Cách 1 đổi mỗi ký tự hai lần).

In [12]:
def score_prev(s: str) -> int:
    prev = ord(s[0])                       # mã của ký tự đầu
    total = 0
    for i in range(1, len(s)):
        cur = ord(s[i])
        total += abs(cur - prev)           # cặp (ký tự trước, ký tự hiện tại)
        prev = cur                         # ký tự hiện tại thành "trước" cho bước sau
    return total


for s, expected in examples:
    print(repr(s), score_prev(s), "(kỳ vọng", expected, ")")

'hello' 13 (kỳ vọng 13 )
'zaz' 50 (kỳ vọng 50 )
'ab' 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**. Chú ý: `n >= 2` theo đề nên `s[0]` luôn tồn tại; nếu `s` rỗng sẽ báo `IndexError` (ngoài ràng buộc).

### Cách 4 — Đổi sang dãy mã rồi dùng `map` + `sub`

**Giải thích.** `codes = s.encode()` cho dãy số (mã) của các ký tự (đúng cho chữ cái ASCII). Hiệu các cặp liền kề: `map(sub, codes, codes[1:])` (mỗi cặp `a − b`); bọc `abs` rồi `sum`: `sum(map(abs, map(sub, codes, codes[1:])))`. Đây là phong cách lập trình hàm.

In [13]:
def score_map(s: str) -> int:
    codes = s.encode()                                  # bytes: duyệt ra các số nguyên
    return sum(map(abs, map(sub, codes, codes[1:])))    # tổng |a - b| trên các cặp liền kề


for s, expected in examples:
    print(repr(s), score_map(s), "(kỳ vọng", expected, ")")

'hello' 13 (kỳ vọng 13 )
'zaz' 50 (kỳ vọng 50 )
'ab' 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (bytes và bản cắt). Lưu ý `s.encode()` chỉ cho "một mã trên một ký tự" khi chuỗi gồm ký tự ASCII (đúng với đề).

### Cách 5 — `itertools.pairwise`

**Giải thích.** `pairwise(s)` cho các cặp ký tự liền kề `(s[0], s[1]), (s[1], s[2]), ...` **không tạo bản sao** của chuỗi (nó là bộ sinh). Cách này cần Python `>= 3.10` (LeetCode hiện hỗ trợ).

In [14]:
def score_pairwise(s: str) -> int:
    return sum(abs(ord(a) - ord(b)) for a, b in pairwise(s))


for s, expected in examples:
    print(repr(s), score_pairwise(s), "(kỳ vọng", expected, ")")

'hello' 13 (kỳ vọng 13 )
'zaz' 50 (kỳ vọng 50 )
'ab' 1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** (bộ sinh). Rất gọn và an toàn về biên.

## 8. TRACE từng bước

Ta trace Cách 1 (chỉ số) và Cách 3 (`prev`) trên `"hello"` và trên `"zaz"`.

In [15]:
def trace_index(s: str) -> int:
    total = 0
    print(f"s = {s!r}, n = {len(s)}, số cặp = {len(s) - 1}")
    for i in range(len(s) - 1):
        a, b = s[i], s[i + 1]
        diff = ord(a) - ord(b)
        total += abs(diff)
        print(f"i={i}: ({a}, {b}) -> ord {ord(a)} - {ord(b)} = {diff:+d}, |.| = {abs(diff)}, tổng = {total}")
    return total


assert trace_index("hello") == 13
print()
assert trace_index("zaz") == 50

s = 'hello', n = 5, số cặp = 4
i=0: (h, e) -> ord 104 - 101 = +3, |.| = 3, tổng = 3
i=1: (e, l) -> ord 101 - 108 = -7, |.| = 7, tổng = 10
i=2: (l, l) -> ord 108 - 108 = +0, |.| = 0, tổng = 10
i=3: (l, o) -> ord 108 - 111 = -3, |.| = 3, tổng = 13

s = 'zaz', n = 3, số cặp = 2
i=0: (z, a) -> ord 122 - 97 = +25, |.| = 25, tổng = 25
i=1: (a, z) -> ord 97 - 122 = -25, |.| = 25, tổng = 50


In [16]:
def trace_prev(s: str) -> int:
    prev = ord(s[0])
    total = 0
    print(f"khởi tạo prev = ord({s[0]!r}) = {prev}")
    for i in range(1, len(s)):
        cur = ord(s[i])
        step = abs(cur - prev)
        total += step
        print(f"i={i}: cur = ord({s[i]!r}) = {cur}, |cur - prev| = {step}, tổng = {total}")
        prev = cur
    return total


assert trace_prev("hello") == 13

khởi tạo prev = ord('h') = 104
i=1: cur = ord('e') = 101, |cur - prev| = 3, tổng = 3
i=2: cur = ord('l') = 108, |cur - prev| = 7, tổng = 10
i=3: cur = ord('l') = 108, |cur - prev| = 0, tổng = 10
i=4: cur = ord('o') = 111, |cur - prev| = 3, tổng = 13


Bảng trace Cách 1 với `"hello"`:

| i | (s[i], s[i+1]) | mã | hiệu | `abs` | tổng |
|---|---|---|---|---|---|
| 0 | (h, e) | 104, 101 | +3 | 3 | 3 |
| 1 | (e, l) | 101, 108 | −7 | 7 | 10 |
| 2 | (l, l) | 108, 108 | 0 | 0 | 10 |
| 3 | (l, o) | 108, 111 | −3 | 3 | 13 |

Với `"zaz"`: `|122 − 97| = 25` rồi `|97 − 122| = 25` ⇒ `50`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (`zip` + `sum`): ngắn gọn, đúng biên, `O(n)`.

In [17]:
class Solution:
    def scoreOfString(self, s: str) -> int:
        # zip(s, s[1:]) ghép mỗi ký tự với ký tự kế tiếp: đúng n - 1 cặp liền kề
        return sum(abs(ord(a) - ord(b)) for a, b in zip(s, s[1:]))


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.scoreOfString(s))

'hello' -> 13
'zaz' -> 50
'ab' -> 1


## 10. Vì sao thuật toán đúng

- **Đúng các cặp:** các cặp liền kề của `s` là `(s[i], s[i+1])` với `i = 0..n-2`. `zip(s, s[1:])` tạo đúng chuỗi cặp đó: phần tử thứ `i` của `s[1:]` là `s[i+1]`, và `zip` dừng khi `s[1:]` (ngắn hơn một phần tử) hết.
- **Đúng đóng góp:** mỗi cặp đóng góp `|ord(a) − ord(b)|` theo định nghĩa đề; `abs` đảm bảo không âm (không trừ lẫn nhau giữa các cặp).
- **Tổng:** `sum` cộng mọi đóng góp, ra điểm của chuỗi.
- **Không có cặp vòng tròn:** đề chỉ nói "hai ký tự liền kề"; ký tự cuối không có ký tự kế tiếp nên không thuộc cặp nào ở vị trí "đầu cặp".
- **Biên:** `n >= 2` nên có ít nhất một cặp; kết quả luôn `>= 0`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp chỉ số | `O(n)` | `O(1)` |
| 2. `zip(s, s[1:])` | `O(n)` | `O(n)` (bản cắt `s[1:]`) |
| 3. Biến `prev` | `O(n)` | `O(1)` |
| 4. `bytes` + `map(sub)` | `O(n)` | `O(n)` |
| 5. `pairwise` | `O(n)` | `O(1)` |

Mọi cách đều `O(n)` thời gian, tối ưu vì phải đọc mọi ký tự.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "index": score_index,
    "zip": score_zip,
    "prev": score_prev,
    "map": score_map,
    "pairwise": score_pairwise,
    "leetcode": sol.scoreOfString,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all("aa", 0, "hai ký tự giống nhau")
check_all("ab", 1, "hai ký tự kề nhau trong bảng chữ cái")
check_all("az", 25, "khoảng cách lớn nhất giữa hai chữ cái")
check_all("za", 25, "cùng khoảng cách, thứ tự ngược (abs)")
check_all("aaa", 0, "ba ký tự giống nhau")
check_all("abc", 2, "tăng dần: 1 + 1")
check_all("cba", 2, "giảm dần: 1 + 1")
check_all("azaz", 75, "zigzag lớn nhất: 25 * 3")
check_all("abcdefghijklmnopqrstuvwxyz", 25, "bảng chữ cái xuôi: 25 bước, mỗi bước 1")
check_all("zyxwvutsrqponmlkjihgfedcba", 25, "bảng chữ cái ngược")
check_all("a" * 100, 0, "n = 100 toàn chữ a")
check_all("az" * 50, 25 * 99, "n = 100 zigzag lớn nhất")

PASS hai ký tự giống nhau
PASS hai ký tự kề nhau trong bảng chữ cái
PASS khoảng cách lớn nhất giữa hai chữ cái
PASS cùng khoảng cách, thứ tự ngược (abs)
PASS ba ký tự giống nhau
PASS tăng dần: 1 + 1
PASS giảm dần: 1 + 1
PASS zigzag lớn nhất: 25 * 3
PASS bảng chữ cái xuôi: 25 bước, mỗi bước 1
PASS bảng chữ cái ngược
PASS n = 100 toàn chữ a
PASS n = 100 zigzag lớn nhất


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **quên `abs`** (các hiệu âm triệt tiêu nhau); (b) **đếm thêm cặp vòng tròn** (ký tự cuối với ký tự đầu).

In [20]:
def wrong_no_abs(s):
    return sum(ord(a) - ord(b) for a, b in zip(s, s[1:]))


def wrong_circular(s):
    return sum(abs(ord(s[i]) - ord(s[(i + 1) % len(s)])) for i in range(len(s)))


bad1 = "zaz"
bad2 = "hello"
print(wrong_no_abs(bad1), "!=", score_zip(bad1))
print(wrong_circular(bad2), "!=", score_zip(bad2))
assert wrong_no_abs(bad1) != score_zip(bad1)
assert wrong_circular(bad2) != score_zip(bad2)

0 != 50
20 != 13


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi đúng ràng buộc (`2..100` ký tự, chữ thường). Tham chiếu độc lập: dựng list mã rồi tính bằng vòng lặp thuần.

In [21]:
def reference(s):
    codes = [ord(c) for c in s]
    total = 0
    for i in range(1, len(codes)):
        d = codes[i] - codes[i - 1]
        total += d if d >= 0 else -d
    return total


rng = random.Random(3110)
for trial in range(3000):
    alphabet = rng.choice(["a", "ab", "az", "abc", string.ascii_lowercase])
    n = rng.choice([2, 3, 5, 10, 50, 99, 100])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    assert 2 <= len(s) <= 100 and s.islower()
    ref = reference(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
print("PASS 3000 chuỗi ngẫu nhiên đúng ràng buộc: mọi cách khớp tham chiếu")

PASS 3000 chuỗi ngẫu nhiên đúng ràng buộc: mọi cách khớp tham chiếu


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trừ trực tiếp hai ký tự (quên `ord`)

In [22]:
def sai_quen_ord(s):
    return sum(abs(a - b) for a, b in zip(s, s[1:]))     # SAI: a, b là chuỗi


try:
    sai_quen_ord("hello")
except TypeError as e:
    print("TypeError:", e)
print("đúng:", Solution().scoreOfString("hello"))

TypeError: unsupported operand type(s) for -: 'str' and 'str'
đúng: 13


**Vì sao sai:** `a` và `b` là **chuỗi** một ký tự; phép trừ không áp dụng cho chuỗi (`TypeError`). Phải đổi sang mã số bằng `ord`. **Cách sửa:** `abs(ord(a) - ord(b))`.

### Lỗi 2: Quên `abs`

In [23]:
def sai_quen_abs(s):
    return sum(ord(a) - ord(b) for a, b in zip(s, s[1:]))


print("sai:", sai_quen_abs("zaz"), "| đúng:", Solution().scoreOfString("zaz"))
print("sai:", sai_quen_abs("hello"), "| đúng:", Solution().scoreOfString("hello"))

sai: 0 | đúng: 50
sai: -7 | đúng: 13


**Vì sao sai:** nếu không lấy `abs`, các hiệu dương và âm **triệt tiêu**: với `"zaz"`: `+25` và `−25` cộng thành `0` thay vì `50`. (Tổng các hiệu liên tiếp chỉ còn là `ord(đầu) − ord(cuối)`: tổng "teo" lại — tính chất *telescoping*.) **Cách sửa:** `abs` cho **từng** cặp trước khi cộng.

### Lỗi 3: Vượt biên chỉ số (`i` chạy hết `n`)

In [24]:
def sai_vuot_bien(s):
    total = 0
    for i in range(len(s)):                         # SAI: i = n - 1 thì s[i + 1] không tồn tại
        total += abs(ord(s[i]) - ord(s[i + 1]))
    return total


try:
    sai_vuot_bien("hello")
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().scoreOfString("hello"))

IndexError: string index out of range
đúng: 13


**Vì sao sai:** ở `i = n − 1` ta đọc `s[n]`, ngoài chuỗi. **Cách sửa:** `range(len(s) - 1)`, hoặc dùng `zip(s, s[1:])` / `pairwise(s)` vốn tự dừng đúng.

### Lỗi 4: Cộng thêm cặp vòng tròn (cuối với đầu)

In [25]:
def sai_vong_tron(s):
    return sum(abs(ord(s[i]) - ord(s[i - 1])) for i in range(len(s)))     # SAI: i = 0 -> s[-1]


print("sai:", sai_vong_tron("hello"), "| đúng:", Solution().scoreOfString("hello"))

sai: 20 | đúng: 13


**Vì sao sai:** `s[i - 1]` với `i = 0` là `s[-1]` (ký tự **cuối**), tức Python âm thầm ghép đầu với cuối. Đề chỉ tính các cặp **liền kề thật sự** (`n − 1` cặp). Kết quả sai là `13 + |104 − 111| = 20`. **Cách sửa:** bắt đầu `i` từ `1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Hai phần tử liền kề**" ⇒ `zip(s, s[1:])` hoặc `pairwise(s)`; mẫu này lặp lại rất nhiều (kiểm tra tăng/giảm, chênh lệch liền kề, ...).
- Ký tự → số: `ord`; số → ký tự: `chr`. Với chữ thường: `ord(c) - ord('a')` là chỉ số `0..25` trong bảng chữ cái.
- Có `abs` thì mới có "khoảng cách"; không có thì tổng các hiệu liền kề **thu gọn** về `đầu − cuối`.
- `n` ký tự ⇒ `n − 1` cặp; đây là nguồn lỗi lệch-một (*off-by-one*) kinh điển.
- Cần `O(1)` bộ nhớ ⇒ dùng biến `prev` hoặc `pairwise` thay vì `s[1:]`.

## 15. Biến thể của bài

1. **Tính điểm theo khoảng cách trong bảng chữ cái, có vòng** (`'a'` và `'z'` kề nhau): dùng `min(d, 26 - d)`.
2. **Điểm cực đại/cực tiểu của một cặp** thay vì tổng.
3. **Hoán vị các ký tự để điểm lớn nhất/nhỏ nhất** (liên quan sắp xếp).
4. **Minimum Absolute Difference (1200)**, **Sum of Absolute Differences in Sorted Array (1685)**.
5. **Chuỗi gồm cả chữ hoa/chữ số**: `ord` vẫn dùng được (bảng mã Unicode).

In [26]:
def score_circular_alphabet(s):
    """Khoảng cách trên vòng tròn bảng chữ cái (a và z kề nhau)."""
    total = 0
    for a, b in zip(s, s[1:]):
        d = abs(ord(a) - ord(b))
        total += min(d, 26 - d)
    return total


def max_pair_diff(s):
    return max(abs(ord(a) - ord(b)) for a, b in zip(s, s[1:]))


print(score_circular_alphabet("zaz"))        # 1 + 1 = 2
print(max_pair_diff("hello"))                # 7
assert score_circular_alphabet("zaz") == 2
assert score_circular_alphabet("hello") == 13
assert max_pair_diff("hello") == 7
assert score_circular_alphabet("az") == 1

2
7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp chỉ số | `s[i]`, `s[i+1]` | `O(n)` | `O(1)` | hiểu rõ từng bước |
| 2. `zip(s, s[1:])` | ghép ký tự với ký tự kề | `O(n)` | `O(n)` | **nên nộp**, gọn nhất |
| 3. Biến `prev` | nhớ ký tự trước | `O(n)` | `O(1)` | cần bộ nhớ nhỏ, dữ liệu dòng |
| 4. `bytes` + `map` | dãy mã, lập trình hàm | `O(n)` | `O(n)` | chỉ khi ký tự là ASCII |
| 5. `pairwise` | cặp liền kề không tạo bản sao | `O(n)` | `O(1)` | Python 3.10+, gọn và an toàn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `ord("c")`, `chr(100)`, `abs(-5 + 2)`, `list(zip("abc", "bc"))`, `Solution().scoreOfString("abd")`.

**Bài 2 (tính tay).** Với `s = "code"`: viết mã của từng ký tự (`c = 99`), các `|hiệu|` liền kề và điểm. Làm tương tự với `"azbz"`.

**Bài 3 (điền TODO).** Hoàn thành `score_todo(s)` theo Cách 3 (biến `prev`).

In [27]:
def score_todo(s):
    # TODO 1: prev = mã của ký tự đầu tiên
    total = 0
    # TODO 2: duyệt i từ 1 tới len(s) - 1
    #   TODO 3: cur = mã của s[i]; total += abs(cur - prev)
    #   TODO 4: cập nhật prev = cur
    # TODO 5: return total
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `score_pairwise2(s)` theo Cách 5 nhưng **không dùng `sum` với generator**: dùng vòng `for a, b in pairwise(s)` và biến cộng dồn.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_score_permutation(s)` trả về **điểm lớn nhất** có thể đạt khi **sắp xếp lại các ký tự** của `s` (thử mọi hoán vị với `len(s) <= 6`, rồi kiểm tra). (b) Viết `min_score_permutation(s)` trả về điểm **nhỏ nhất** có thể đạt khi sắp xếp lại; chứng minh/nghiệm lại rằng nó bằng `ord(max) - ord(min)` (sắp xếp tăng dần).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `99`, `'d'`, `3`, `[('a', 'b'), ('b', 'c')]`, `3` (`|97 − 98| + |98 − 100| = 1 + 2`). Bài 3: `prev = ord(s[0])`, `for i in range(1, len(s))`.
- **Mức 2 (vừa):** Bài 2: `code`: mã `99, 111, 100, 101`; `|99 − 111| = 12`, `|111 − 100| = 11`, `|100 − 101| = 1` ⇒ `24`. `azbz`: `25 + 24 + 24 = 73`. Bài 4: `total = 0`, `for a, b in pairwise(s): total += abs(ord(a) - ord(b))`.
- **Mức 3 (sâu):** Bài 5(a): dùng `itertools.permutations(s)` và `max(score(...))`. 5(b): với các ký tự đã sắp xếp tăng dần, hiệu liền kề không âm nên `abs` không đổi dấu, tổng thu gọn thành `ord(max) - ord(min)`; đây là điểm nhỏ nhất vì mọi đường đi qua mọi ký tự phải đi ít nhất từ min tới max.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from itertools import permutations

# Bài 1
assert ord("c") == 99
assert chr(100) == "d"
assert abs(-5 + 2) == 3
assert list(zip("abc", "bc")) == [("a", "b"), ("b", "c")]
assert Solution().scoreOfString("abd") == 3

# Bài 2
assert [ord(c) for c in "code"] == [99, 111, 100, 101]
assert Solution().scoreOfString("code") == 12 + 11 + 1
assert Solution().scoreOfString("azbz") == 25 + 24 + 24


# Bài 3
def score_todo_ans(s):
    prev = ord(s[0])
    total = 0
    for i in range(1, len(s)):
        cur = ord(s[i])
        total += abs(cur - prev)
        prev = cur
    return total


for s, expected in examples:
    assert score_todo_ans(s) == expected


# Bài 4
def score_pairwise2(s):
    total = 0
    for a, b in pairwise(s):
        total += abs(ord(a) - ord(b))
    return total


for s, expected in examples:
    assert score_pairwise2(s) == expected


# Bài 5
def max_score_permutation(s):
    return max(score_index("".join(p)) for p in permutations(s))


def min_score_permutation(s):
    return min(score_index("".join(p)) for p in permutations(s))


assert max_score_permutation("abc") == 3       # a c b -> 2 + 1 ... hoặc b a c: 1 + 2 = 3
assert min_score_permutation("abc") == 2       # abc: 1 + 1
for word in ["hello", "zaz", "code", "abcd", "azbz"]:
    assert min_score_permutation(word) == ord(max(word)) - ord(min(word)), word
    assert min_score_permutation(word) == score_index("".join(sorted(word)))
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng `|hiệu|` của các cặp liền kề)

```python
class Solution:
    def scoreOfString(self, s: str) -> int:
        return sum(abs(ord(a) - ord(b)) for a, b in zip(s, s[1:]))
```

**Checklist**

- [ ] Đổi ký tự sang mã bằng `ord` trước khi trừ?
- [ ] Có `abs` cho **từng** cặp (không phải cho tổng)?
- [ ] Chỉ `n − 1` cặp liền kề: không vượt biên, không ghép cuối với đầu?
- [ ] Đã thử `"aa"` (điểm `0`), `"az"` (`25`), `"zaz"` (`50`)?
- [ ] Khi muốn `O(1)` bộ nhớ: dùng `prev` hoặc `pairwise`?
- [ ] Skeleton đúng `class Solution` / `def scoreOfString(self, s: str) -> int`?